# Cadence — Streak Retention Analysis

Narrative walkthrough of the daily SIP habit analysis.

**This notebook contains no analytical logic.** Every calculation is imported from
`src/`, which exists so the same code runs here, in the scheduled report, and in
CI. A notebook that reimplements its own version of a metric is how a dashboard
and a deck end up disagreeing.

Read [`MEMO.md`](../MEMO.md) if you only want the conclusion.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

import pandas as pd

from src import db
from src.analysis import (
    cohort_analysis,
    data_quality,
    nudge_simulation,
    streak_builder,
    survival_analysis,
    viz,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

## 1. What we're working with

Raw transactions, and the same table after the cleaning rules in
`v_clean_transactions` are applied.

In [ ]:
db.read_sql("""
    SELECT 'users'                  AS table_name, COUNT(*) AS rows FROM users
    UNION ALL SELECT 'transactions (raw)',   COUNT(*) FROM sip_daily_transactions
    UNION ALL SELECT 'transactions (clean)', COUNT(*) FROM v_clean_transactions
    UNION ALL SELECT 'streaks',              COUNT(*) FROM user_streaks
    UNION ALL SELECT 'nudges sent',          COUNT(*) FROM nudges_sent
""")

The gap between raw and clean is not noise — it is exactly the duplicate and
pre-signup rows seeded by the generator. Section 6 quantifies them.

## 2. Streaks

A streak is a run of consecutive days with a successful contribution. Built with
the gaps-and-islands pattern: within an unbroken run, `txn_date - ROW_NUMBER()`
is constant, so that difference labels the streak.

The SQL and pandas implementations are asserted equal — drift between them would
be a test failure, not a surprise six months from now.

In [ ]:
streaks = streak_builder.load_streaks()
print(f"{len(streaks):,} streaks across {streaks['user_id'].nunique():,} users")
streaks.head()

In [ ]:
streaks.groupby("archetype").agg(
    streaks=("streak_id", "size"),
    avg_length=("streak_length", "mean"),
    max_length=("streak_length", "max"),
    recovery_rate=("recovered", "mean"),
).round(3).sort_values("avg_length", ascending=False)

Note `weekday_only`: **short** streaks but a **high** recovery rate. A weekend
gap is not churn. Streak length alone is a misleading health signal — recovery
rate is what separates a pause from a death.

## 3. Survival analysis

Two different questions, and conflating them is the most common error here:

1. **How long does a streak live?** Streaks still running when the data ends are
   *censored*, not dead.
2. **Once it breaks, does the user come back?** A user still absent at the window's
   edge hasn't returned *yet* — which is not the same as never.

In [ ]:
kmf_streak = survival_analysis.fit_streak_survival(streaks)
print(f"median streak length: {kmf_streak.median_survival_time_:.0f} days")

pd.DataFrame({
    "day": [3, 7, 14, 30, 60],
    "still alive": [float(kmf_streak.predict(d)) for d in [3, 7, 14, 30, 60]],
}).round(3)

In [ ]:
observation_end = pd.to_datetime(
    db.read_sql("SELECT MAX(txn_date) AS d FROM v_clean_transactions")["d"].iat[0]
)
gaps = survival_analysis.build_gap_frame(streaks, observation_end)
kmf_recovery = survival_analysis.fit_recovery(gaps)

recovery = survival_analysis.recovery_by_gap_length(kmf_recovery)
recovery

**This is the central table.** Recovery holds up through day 3, then collapses:
between day 3 and day 7 the chance of losing someone roughly triples.

These are conditional probabilities read off the fitted curve, not row counts.
Counting rows would conflate "never returned" with "hasn't returned yet" and
overstate churn.

### What raises the risk of a break?

Cox proportional hazards, **clustered on `user_id`** — 50k streaks come from 5k
users, and treating them as independent would shrink every standard error to
match a sample size we don't have.

In [ ]:
cox_frame = survival_analysis.build_cox_frame(streaks)
cph = survival_analysis.fit_cox(cox_frame)
survival_analysis.hazard_ratios(cph)

Only `is_first_streak` survives clustering: a user's **first** streak breaks ~18%
faster than their later ones. Channel and city tier show no resolvable effect —
reported as such rather than quietly dropped.

## 4. Cohort retention, redefined

The D1/D7/D30/D90 convention is kept. What changes is the meaning of *retained*:
still transacting, not still holding an account.

In [ ]:
activity, obs_end = cohort_analysis.load_activity()
users = cohort_analysis.load_users()
table = cohort_analysis.build_retention_table(activity, users, obs_end)
cohort_analysis.overall_retention(table)

A conventional chart would quote the D30 number as ~55%. Measured against the
actual daily promise it is ~36% — a 19-point gap between the framing and the
product.

## 5. Did the nudge work?

Arms were randomised at signup, before any behaviour was observed. Each treated
break is compared only against control breaks that reached the **same gap
length** — conditioning on the risk set, the same device Kaplan-Meier uses.

In [ ]:
nudge_results = nudge_simulation.run()
nudge_results

Day 5 is roughly twice as efficient as any other trigger: ~13 sends per extra
recovery against 26–27 elsewhere. Day 1 and 2 nudges land on users already at a
91% baseline — there is very little left to win.

This reproduces the survival curve's day-5 inflection from entirely separate
arithmetic, which is the strongest form of confirmation available here.

## 6. Data quality

Checks read the **raw** tables, never the cleaned view — the view protects the
analysis, these checks protect the ledger.

In [ ]:
findings = data_quality.run_all()
data_quality.summary_frame(findings)

See [`data_quality_findings.md`](../data_quality_findings.md) for detection
queries, blast radius, and the DDL that closes each one.

DQ-05 is the one worth escalating rather than patching: ₹10.7 lakh moved through
accounts the system believes are unverified, and the data cannot distinguish a
missing KYC gate from a stale status field.

## 7. The surprise

Consistency is **bimodal**. There is no meaningful middle of the distribution.

In [ ]:
consistency = db.read_sql("SELECT * FROM v_user_consistency")
buckets = pd.cut(consistency["active_day_ratio"], bins=[0, .1, .2, .3, .5, .7, .9, 1.0])
consistency.groupby(buckets, observed=True).size().rename("users").to_frame()

The mean active-day ratio is ~0.30, and it describes essentially **no real
user**. Any target framed as "raise average consistency" aims at a gap in the
distribution.

The better question is what moves someone from the left cluster to the right one
— and that is what the next experiment should be designed around.

---

## Where this lands

1. **Move the nudge trigger to day 5** — ~660 additional recovered users a year on
   the same send budget.
2. **Instrument day 3 to day 7 properly** — where retention is actually decided,
   and currently least visible.
3. **Design the next experiment around the bimodal split.**

Full reasoning and caveats in [`MEMO.md`](../MEMO.md) and
[`ASSUMPTIONS.md`](../ASSUMPTIONS.md).